In [ ]:
# Add relevant Jupyter notebook extensions 

In [ ]:
# You can double-check your Python path like this...
import sys
import os

# Get the project root — go one level up from the current folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
sys.path.append('../uuv_mission')
print(sys.path)

# Simulate closed-loop
After implementing your control functionality, you can simulate the closed-loop with code that looks something like this...

In [ ]:
# Import relevant modules

from uuv_mission.control import PDController
from uuv_mission.dynamic import ClosedLoop, Mission, Submarine


sub = Submarine()
# Instantiate your controller (depending on your implementation)
controller = PDController()
closed_loop = ClosedLoop(sub, controller)
mission = Mission.from_csv("../data/mission.csv") # You must implement this method in the Mission class

trajectory = closed_loop.simulate_with_random_disturbances(mission)
trajectory.plot_completed_mission(mission)

To test which proportional gain will ensure the smallest error between the reference and the actual trajectory, we run a simulation over a range of possible values.
Based on the simulation we choose the best gain and plot a completed mission.

The parameter we are trying to minimise is the biggest error between the trajectory and the reference. This is done so that we can ensure that the uuv will not hit the cave.
After running the simulation, we can see that the most effective parameter set is Kp = 0.1, Kd = 0.7

In [ ]:
from uuv_mission.control import PDController
from uuv_mission.dynamic import ClosedLoop, Mission, Submarine
import numpy as np


NMB_SIMULATIONS = 20000

def calculate_biggest_difference(trajectory: np.ndarray, reference: np.ndarray) -> float:
    """
    Calculate the biggest difference between the uuv trajectory and the reference trajectory.
    Args:
        trajectory (np.ndarray): The trajectory obtained from the simulation.
        reference (np.ndarray): The reference trajectory from the mission.
    Returns:
        float: The biggest difference between the trajectory and the reference trajectory.
    """

    differences = trajectory - reference
    return differences.max()


def run_simulations(Kp: float, Kd: float) -> float:
    """
    Run multiple simulations with random disturbances and return the maximum difference
    Args:  
        Kp (float): Proportional gain
        Kd (float): Differential gain
    Returns:
        float: The maximum difference observed across all simulations
    """

    controller = PDController(KP=kp, KD=kd)
    closed_loop = ClosedLoop(sub, controller)
    
    max_curr_diff = float('-inf')
    for _ in range(NMB_SIMULATIONS):
        trajectory = closed_loop.simulate_with_random_disturbances(mission)
        biggest_difference = calculate_biggest_difference(trajectory.position[:, 1], mission.reference)
        max_curr_diff = max(max_curr_diff, biggest_difference)

    return max_curr_diff


sub = Submarine()
mission = Mission.from_csv("../data/mission.csv")

# Values of gains to test
proportional_gains = np.linspace(0.05, 0.15, 11)
differential_gains = np.linspace(0.4, 0.8, 11)

min_diff = float('+inf')
final_gains = [0.0, 0.0] # Gains with the smallest difference
for kp in proportional_gains:
    for kd in differential_gains:
        max_curr_diff = run_simulations(kp, kd)

        if max_curr_diff < min_diff:
            min_diff = max_curr_diff
            final_gain = [kp, kd]

print(f"The most stable PD contoller has a proportional proportional gain of {final_gain[0]} and differential gain of {final_gain[1]} with a biggest difference of {min_diff:.2f}")

controller = PDController(KP=final_gain[0], KD=final_gain[1])
closed_loop = ClosedLoop(sub, controller)
trajectory = closed_loop.simulate_with_random_disturbances(mission)
trajectory.plot_completed_mission(mission)